# Paso 1 de la climatología de ITCHI: periodos de retorno de la lluvia

**Qué se calcula.** Para cada píxel de MSWEP V3 (0.1°, mayo–noviembre, 2000–2025) y para cada duración (6, 24 y 72 h), se calcula **cada cuánto se alcanza** una cantidad de lluvia. Se usan todas las lluvias, no solo las de ciclones, porque el peligro de un ciclón se mide contra lo que es raro en ese lugar.

**Por qué.** El P95 de 6 h que usamos hasta ahora es el valor que supera el 5 % de las ventanas *con lluvia* del píxel. Ese número no dice cuántas veces por temporada ocurre, porque depende de cuánto llueve ahí. El periodo de retorno sí lo dice, y en las mismas unidades en todo el país.

## Cómo se lee un periodo de retorno

- **T = 10 años** para 150 mm en 24 h significa que, en ese píxel, la lluvia máxima de 24 h de una temporada supera 150 mm con probabilidad 1/10. En 26 temporadas se espera unas 2 o 3 veces.
- **Se calcula de dos maneras independientes:**
  1. **Máximos estacionales + GEV.** Se toma la lluvia máxima de cada temporada (26 valores por píxel) y se ajusta una distribución de valores extremos (GEV) por L-momentos. Sirve para T ≥ 2 años.
  2. **Picos independientes + GPD (POT).** Se separan los eventos de lluvia (picos con al menos 48 h entre sí; 96 h para 72 h), se toman los 78 mayores del registro (3 por temporada) y se ajusta una distribución de Pareto generalizada. Da el **intervalo medio R** entre eventos y sirve también por debajo de un año: R = 0.33 años equivale a 3 eventos por temporada.
- **Relación entre las dos:** T = 1 / (1 − e^(−1/R)). Por ejemplo, R = 1 año da T = 1.6 años, y R = 10 años da T ≈ 10.5 años.

**Entradas.** `data/processed/mswep_v3_return_periods_may_nov.nc`, generado con:

```bat
conda activate itchi
cd scripts
python periodos_retorno.py        :: ~30 min; lee una vez los 26 años de D:/MSWEP_V316_6H
```

También usa `mswep_v3_{6,24,72}h_percentiles_may_nov.nc`, para comparar con los percentiles actuales.

**Salidas.** Figuras en `outputs/figures/periodos_retorno/`.

## 0. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from itchi.climatology import quantile_dataarray_to_dict
from itchi.config import get_project_root

ROOT = get_project_root()
sys.path.insert(0, str(ROOT / "scripts"))
import figuras_retorno as F      # noqa: E402

RP_FILE = ROOT / "data" / "processed" / "mswep_v3_return_periods_may_nov.nc"
PCT_FILE = ROOT / "data" / "processed" / "mswep_v3_{d}h_percentiles_may_nov.nc"
FIG = ROOT / "outputs" / "figures" / "periodos_retorno"
FIG.mkdir(parents=True, exist_ok=True)
assert RP_FILE.exists(), f"Falta {RP_FILE}: corre primero scripts/periodos_retorno.py"


def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=200, bbox_inches="tight")
    plt.show()


rp = F.load_rp(RP_FILE)
print(f"malla {rp.lat.size} x {rp.lon.size} | duraciones {rp.durations} h | temporadas {rp.seasons[0]}-{rp.seasons[-1]}")


def percentiles_en_malla(d):
    with xr.open_dataset(str(PCT_FILE).format(d=d)) as ds:
        q = ds["p_wet_raw"].load()
    q = q.reindex(lat=rp.lat, lon=rp.lon, method="nearest", tolerance=1e-3)
    return {k: v.transpose("lat", "lon").values for k, v in quantile_dataarray_to_dict(q).items()
            if k in ("Q75", "Q90", "Q95", "Q99")}


PCT = {d: percentiles_en_malla(d) for d in rp.durations}

## R1. Cómo leer un periodo de retorno, en seis ciudades

Cada punto es la lluvia máxima de 24 h de una temporada, ubicada en su periodo de retorno empírico. La línea azul es la GEV y la naranja discontinua es el POT, convertido a periodo anual.

**Cómo leerla.** Si las dos curvas pasan por los puntos y coinciden entre sí hasta unos 10 años, el resultado es robusto. Más allá de 26 años no hay datos: las curvas son extrapolación y pueden divergir. Las líneas verticales marcan T = 2 y T = 10 años.

In [ ]:
for d in rp.durations:
    guardar(F.fig_r1_curva_pixel(rp, duration=d), f"R1_curva_ciudades_{d}h")

## R2. Mapas: lluvia con periodo de retorno de 2 y 10 años

Por duración, la lluvia que se supera en promedio una vez cada 2 años (arriba) y cada 10 años (abajo).

**Cómo leerla.** Estos son los umbrales absolutos que harían «raro» un evento en cada lugar. La vertiente del Pacífico sur y el Golfo deberían tener niveles mucho mayores que el altiplano y el noroeste. Por eso el peligro se define en periodo de retorno y no en milímetros.

In [ ]:
guardar(F.fig_r2_mapas_niveles(rp), "R2_mapas_niveles")

## R3. ¿Qué tan raros son los percentiles que usa ITCHI v0.2?

Para cada píxel, cada cuánto se alcanza su propio P95 y P99 (intervalo medio entre eventos, POT).

**Cómo leerla.** Si el P95 cae en «< 0.33 años», se alcanza tres o más veces por temporada. Llamarlo «peligro máximo» significaría eso. Esta figura es la base para decidir los umbrales de los niveles en periodo de retorno.

In [ ]:
fig, medianas = F.fig_r3_rareza_percentil(rp, PCT)
guardar(fig, "R3_rareza_percentiles")

## R4. Consistencia GEV frente a POT

Nivel de T = 10 años con los dos métodos, píxel por píxel.

**Cómo leerla.** La nube debe pegarse a la diagonal roja. Si se separa en alguna zona, revisar ahí la forma de la cola (R5) o la separación entre eventos.

In [ ]:
guardar(F.fig_r4_consistencia(rp), "R4_consistencia_gev_pot")

## R5. Control: forma de la cola

El parámetro ξ de la GEV y de la GPD. Si ξ > 0, la cola es pesada: los extremos crecen rápido con T.

**Cómo leerla.** Con 26 temporadas, ξ por píxel es muy incierto. Por eso está acotado entre −0.5 y 0.5. Si muchos píxeles quedan acotados o el mapa es puro ruido, los niveles de 25 años o más no son confiables y conviene un ajuste regional.

In [ ]:
guardar(F.fig_r5_forma(rp), "R5_forma_cola")

## Resumen en México

Medianas sobre la caja de México: lluvia con periodo de retorno T e intervalo de recurrencia de los percentiles actuales.

In [ ]:
tabla = F.tabla_resumen(rp, PCT)
tabla.to_csv(FIG / "resumen_periodos_retorno.csv")
tabla.round(2)

## Siguiente paso

Con estos parámetros, cualquier lluvia (P6, R24, A72) de cualquier ciclón se convierte en su periodo de retorno con `itchi.return_periods.gev_return_period` y `pot_recurrence_interval`. Ese es el paso 2: el catálogo de estados bajo los ciclones de 2000 a 2025, en malla y por municipio, sobre el que se prueban los esquemas de niveles.